In [1]:
!pip install -q -U torchao peft transformers huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.5/661.5 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 58.7 MB/s eta 0:00:00


In [2]:
import torch
import gc
import os
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import login, HfApi
from kaggle_secrets import UserSecretsClient

# 1. Hugging Face Authentication
try:
    user_secrets = UserSecretsClient()
    HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
    
    if HF_TOKEN:
        print("Logged in successfully to Hugging Face.")
        login(HF_TOKEN)
    else:
        # Fallback if secret is found but empty
        raise ValueError("HF_TOKEN secret is empty.")
except Exception as e:
    print(f"❌ Error: Could not retrieve HF_TOKEN from Kaggle Secrets. {e}")
    print("Make sure you added the secret and ENABLED it for this notebook.")

# 2. Path Configurations
ADAPTER_PATH = "/kaggle/input/models/tranphungdinh/finetuned-llama-3-2-1b-instruct/pytorch/dora/1" 
HF_REPO_ID = "tpdinh2612/Llama-3.2-1B-Banking77-Intent-Classification"

print("--- Initializing Merge & Push Process ---")

# 3. Load Base Model
print("Step 1: Loading Base Model on CPU...")
base_model = AutoModelForCausalLM.from_pretrained(
    "unsloth/Llama-3.2-1B-Instruct",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    device_map="cpu", 
)

# 4. Load DoRA Adapter and Merge
print("Step 2: Loading DoRA Adapter and merging weights...")
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
merged_model = model.merge_and_unload()

# 5. Load Tokenizer
print("Step 3: Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)

# 6. Push to Hugging Face Hub
print(f"Step 4: Pushing model to Hub: {HF_REPO_ID}...")

# Manually create repo first to verify permissions
api = HfApi()
try:
    api.create_repo(repo_id=HF_REPO_ID, token=HF_TOKEN, exist_ok=True)
    
    # Push the model
    merged_model.push_to_hub(
        HF_REPO_ID, 
        token=HF_TOKEN,
        max_shard_size="2GB"
    )
    
    # Push the tokenizer
    tokenizer.push_to_hub(
        HF_REPO_ID, 
        token=HF_TOKEN
    )
    print(f"✅ Success! Model is live at: https://huggingface.co/{HF_REPO_ID}")

except Exception as e:
    print(f"❌ Failed to push: {e}")
    print("Double-check if your token is 'WRITE' type and the username matches your HF account.")

# 7. Memory Management
print("Step 5: Cleaning up memory...")
del merged_model, base_model, model
gc.collect()
torch.cuda.empty_cache()

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cpu).


Logged in successfully to Hugging Face.
--- Initializing Merge & Push Process ---
Step 1: Loading Base Model on CPU...


config.json:   0%|          | 0.00/894 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

...

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✅ Success! Model is live at: https://huggingface.co/tpdinh2612/Llama-3.2-1B-Banking77-Intent-Classification
Step 5: Cleaning up memory...
